In [1]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import xarray as xr
from shapely import wkt

from pathlib import Path


In [2]:
bath_url = "https://dap.ceda.ac.uk/thredds/dodsC/bodc/gebco/global/gebco_2026/ice_surface_elevation/netcdf/GEBCO_2026.nc"

# my_session = session=auth.get_session()
bath_ds = xr.open_dataset(bath_url)

In [3]:
bath_ds

<xarray.Dataset> Size: 7GB
Dimensions:    (lat: 43200, lon: 86400)
Coordinates:
  * lat        (lat) float64 346kB -90.0 -89.99 -89.99 ... 89.99 89.99 90.0
  * lon        (lon) float64 691kB -180.0 -180.0 -180.0 ... 180.0 180.0 180.0
Data variables:
    crs        |S64 64B ...
    elevation  (lat, lon) int16 7GB ...
Attributes: (12/37)
    title:                           The GEBCO_2026 Grid - a continuous terra...
    summary:                         The GEBCO_2026 Grid is a continuous, glo...
    keywords:                        BATHYMETRY/SEAFLOOR TOPOGRAPHY, DIGITAL ...
    Conventions:                     CF-1.6, ACDD-1.3
    id:                              DOI: 10.5285/4f68d5c7-45eb-f999-e063-708...
    naming_authority:                https://dx.doi.org
    ...                              ...
    geospatial_vertical_resolution:  1.0
    geospatial_vertical_positive:    up
    identifier_product_doi:          DOI: 10.5285/4f68d5c7-45eb-f999-e063-708...
    references:                      DOI: 10.5285/4f68d5c7-45eb-f999-e063-708...
    node_offset:                     1.0
    DODS.strlen:                     0

In [4]:

station_location_resource_code = "cjfb-f4d4"
url = f"https://data.novascotia.ca/resource/{station_location_resource_code}.csv"
df = pd.read_csv(url)
df = df[df['latitude'].between(44,45.25)& df['longitude'].between(-66.75,-65.72)]
gdf = gpd.GeoDataFrame(df, geometry=gpd.GeoSeries.from_wkt(df['geocoded_column']), crs='EPSG:4326')

# df.head()

In [5]:
# Extract lon/lat arrays from GeoPandas geometry
lons = xr.DataArray(gdf.geometry.x, dims="points")
lats = xr.DataArray(gdf.geometry.y, dims="points")

# Select nearest grid cells
station_elevation = bath_ds['elevation'].sel(lon=lons, lat=lats, method="nearest")

In [6]:
gdf['elevation'] = station_elevation.values

In [7]:
gdf

,county,waterbody,station,latitude,longitude,geocoded_column,geometry,elevation
28,Digby,St. Marys Bay,1012,44.4071,-66.1605,POINT (-66.1605 44.4071),POINT (-66.1605 44.4071),-21
29,Digby,St. Marys Bay,5007,44.5391,-65.9580,POINT (-65.958 44.5391),POINT (-65.958 44.5391),-5
30,Digby,St. Marys Bay,5008,44.5038,-65.9324,POINT (-65.9324 44.5038),POINT (-65.9324 44.5038),-6
31,Digby,St. Marys Bay,Centreville 1,44.5328,-65.9840,POINT (-65.984 44.5328),POINT (-65.984 44.5328),-8
32,Digby,St. Marys Bay,Centreville,44.5362,-66.0017,POINT (-66.0017 44.5362),POINT (-66.0017 44.5362),-6
33,Digby,St. Marys Bay,Church Point 1,44.3240,-66.1357,POINT (-66.1357 44.324),POINT (-66.1357 44.324),-10
34,Digby,St. Marys Bay,Church Point 2,44.3202,-66.1351,POINT (-66.1351 44.3202),POINT (-66.1351 44.3202),-9
35,Digby,St. Marys Bay,Long Beach 1,44.3818,-66.1774,POINT (-66.1774 44.3818),POINT (-66.1774 44.3818),-39
36,Digby,St. Marys Bay,Long Beach,44.4017,-66.1596,POINT (-66.1596 44.4017),POINT (-66.1596 44.4017),-33
37,Digby,St. Marys Bay,Long Island 1,44.3501,-66.2243,POINT (-66.2243 44.3501),POINT (-66.2243 44.3501),-33


In [8]:
gdf.to_csv("stations_with_elevation.csv")
